In [ ]:
!pip install groq chromadb pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 95.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 107.4 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Su

In [ ]:
from groq import Groq
from google.colab import userdata
api_key = userdata.get('ChatBotKey')
client=Groq(api_key=api_key)
MODEL = "openai/gpt-oss-120b"

In [ ]:
import os
os.makedirs("docs",exist_ok=True)

In [ ]:
print(os.listdir("docs"))

['.ipynb_checkpoints', 'Trippy_AI.pdf']


STEP : 01 (PDF Loading)

In [ ]:
import re
from pypdf import PdfReader
def load_pdf(path):
  pages = []
  reader = PdfReader(path)
  for number, page in enumerate(reader.pages, start=1):
    pages.append({"text": page.extract_text() or " ", "source": os.path.basename(path), "page": number})
  return pages

In [ ]:
def load_folder(folder):
  pages = []
  for name in sorted(os.listdir(folder)):
    path=os.path.join(folder,name)
    if name.lower().endswith(".pdf"):
      pages+=load_pdf(path)

  return pages

STEP 02 (Cleaning & Chunking)

In [ ]:
def txt_cleaning(text):
  return " ".join(text.split())

def chunk_sentence(text, max_char=400):
    sentences = re.split(r'(?<=[.!?])\s+|\n+', text) # split after . ! or ?

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if not sentence.strip():
            continue
        if current_chunk and len(current_chunk) + len(sentence) + 1 > max_char:
            chunks.append(current_chunk)
            current_chunk = sentence
        else:
            current_chunk = (current_chunk + " " + sentence).strip() if current_chunk else sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

def chunk_paragraph(text, max_chars=400):
    """Cut at blank lines first. Cut a paragraph into sentences only if it is longer than max_chars."""
    chunks = []
    for paragraph in text.split("\n\n"):
        paragraph = txt_cleaning(paragraph)
        if not paragraph:
            continue


        if len(paragraph) <= max_chars:
            chunks.append(paragraph)
        else:
            chunks.extend(chunk_sentence(paragraph, max_chars))
    return chunks

def build_chunks(folder,max_chars=400):
  all_chunks = []
  for page in load_folder(folder):
    if not page["text"].strip():
      print(f"Skipped empty page (a scan? needs OCR):", page["source"], "page:", page["page"])
      continue
    for piece in chunk_paragraph(page["text"],max_chars):
      all_chunks.append({"text":piece,"source":page["source"],"page":page["page"]})
  return all_chunks

In [ ]:
chunks = build_chunks("docs")
print(len(chunks), "Chunks :")
print(chunks[0])

113 Chunks :
{'text': 'Trippy AI — Project Report Page 1 1. Introduction Travel planning often requires people to collect information from many sources. A traveller may need to choose a destination, estimate costs, prepare a daily schedule, check the weather, compare transport options, and find accommodation. When this information is spread across separate websites, planning can become confusing and time-consuming.', 'source': 'Trippy_AI.pdf', 'page': 1}


STEP 03: EMBEDDINGS & STORING CHUNKS IN CHROMADB

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embed = DefaultEmbeddingFunction()
chroma = chromadb.Client()

def store_chunks(chunks,name="docs_kb"):
  try:
    chroma.delete_collection(name)
  except Exception:
    pass
  collection=chroma.create_collection(name=name,embedding_function=embed)
  collection.upsert(
      documents=[c["text"] for c in chunks],
      metadatas=[{"source":c["source"],"page":c["page"]} for c in chunks],
      ids=[f"chunk{i}" for i in range(len(chunks))]
  )
  return collection
collection=store_chunks(chunks)
print("Stored:",collection.count(),"chunks")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 10.2MiB/s]


Stored: 113 chunks


STEP NO 04: RETRIEVE

In [ ]:
def retrieve(question,n=3):
  result=collection.query(query_texts=[question],n_results=min(n,collection.count()))
  found=[]
  for text,meta,dist in zip(result["documents"][0],result["metadatas"][0],result['distances'][0]):
    found.append({"text":text,"source":meta["source"],"page":meta["page"],"distance":dist})
    return found
for item in retrieve("What is Trippy AI?"):
    print(f'{item["distance"]:2f} | {item["source"]} page {item["page"]} | {item["text"][:60]}...')

0.372093 | Trippy_AI.pdf page 9 | Conclusion Trippy AI is a Python and Streamlit travel assist...


STEP 05: ANSWER WITH MEMORY

In [ ]:

RULES = (
    "You answer questions using ONLY the context provided. "
    "If the answer is not in the context, reply exactly: I don't know."
)

chat_history = []


def ask(question, found, history):
    context = "\n\n".join(
        f"[{c['source']}, page {c['page']}]\n{c['text']}"
        for c in found
    )

    messages = [{"role": "system", "content": RULES}]
    messages += history

    messages.append({
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {question}"
    })

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
        max_tokens=2000
    )

    return response.choices[0].message.content


def rag_chat(question):
    found = retrieve(question)

    reply = ask(question, found, chat_history[-6:])

    chat_history.append({
        "role": "user",
        "content": f"Question: {question}\nRetrieved context: "
                   + "\n".join(c["text"] for c in found)
    })
    chat_history.append({
        "role": "assistant",
        "content": reply
    })

    return reply, found


def interactive_chat():
    print("****** Welcome to Trippy AI ******")
    print("Ask questions about Trippy AI.")
    print("Type 'exit' to stop the chatbot.\n")

    while True:
        question = input("You: ").strip()

        if question.lower() in ["exit", "quit", "bye"]:
            print("Chatbot: Goodbye!")
            break

        if not question:
            print("Please enter a question.\n")
            continue

        try:
            reply, found = rag_chat(question)

            print(f"\nChatbot: {reply}")

            if found:
                print("\nSources:")
                for c in found:
                    print(f"- {c['source']} (Page {c['page']})")
            else:
                print("\nNo sources found.")

            print()

        except Exception as e:
            print(f"An error occurred: {e}\n")


interactive_chat()


****** Welcome to Trippy AI ******
Ask questions about Trippy AI.
Type 'exit' to stop the chatbot.

You: give me budget recommendation

Chatbot: I don't know.

Sources:
- Trippy_AI.pdf (Page 4)

You: budget recommendation

Chatbot: Based on the provided information, a solid budget recommendation for your trip should include estimates for each of the major expense categories mentioned:

1. **Transport** – budget for flights or other long‑distance travel to your destination, plus local transportation (buses, trains, taxis, car rentals, etc.).  
2. **Accommodation** – estimate the nightly cost of hotels, hostels, vacation rentals, or other lodging options for the entire stay.  
3. **Meals** – plan for daily food expenses, whether you’ll dine out, buy groceries, or a mix of both.  
4. **Sightseeing** – include entry fees, guided tours, and any other costs associated with visiting attractions.  
5. **Local Activities** – set aside money for optional experiences such as adventure sports, cul